# `build_pptx_document` — rendering PPTX (step 2 du build order tome 2)

Module testé : [`src/docpipeline/rendering/pptx/build_document.py`](../src/docpipeline/rendering/pptx/build_document.py).

## Le pattern transversal du projet

```
extract  : parse_pptx(source)              → runs_df avec span_id stable
modify   : on remplace .text de chaque run par le texte traduit
rebuild  : build_pptx_document(runs_df, source, output)
           → ouvre source comme template, walk slides/shapes/runs,
             replace .text par span_id, save
```

Le `span_id` (format `pp_<slide>_<shape>_<para>_<run>`) est la clé stable, cohérente avec le `w_<para>_<run>` Word et le `p_<page>_<line>` PDF.

## Démo en 2 temps

1. **Round-trip identité** — `parse_pptx(source) → build_pptx_document(runs_df, source, out)` doit produire un .pptx visuellement identique à la source. Smoke test du tome 2 step 2.
2. **Traduction FR → EN** — on remplace le texte de chaque run par sa traduction (via dictionnaire métier, sans LLM dans la démo). Le .pptx traduit est sauvegardé dans `notebooks/_outputs/pptx_translation/` pour inspection visuelle dans PowerPoint.

Aucun LLM (règle CLAUDE.md). En prod, le step `translate_chunks` (LLM) viendra entre extract et rebuild ; ici on simule par un dictionnaire.

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
# 1) Round-trip identité (smoke test step 2)
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from docpipeline.parsing.pptx.parse_pptx import parse_pptx
from docpipeline.rendering.pptx.build_document import build_pptx_document

SRC          = Path('../tests/fixtures/contrat_assurance.pptx')
OUT_ROUND    = Path('_outputs/pptx_translation/contrat_assurance_roundtrip_js.pptx')
OUT_TRANS_FR = Path('_outputs/pptx_translation/contrat_assurance_EN_js.pptx')
OUT_ROUND.parent.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_colwidth', 100)

# Parse
result_src = parse_pptx(SRC)
runs_df = result_src['runs_df']

# Round-trip : passer le runs_df tel quel
rt_result = build_pptx_document(runs_df, SRC, OUT_ROUND)

display(Markdown('### 1. Round-trip identité — smoke test step 2'))
display(pd.DataFrame([{
    'metric': 'runs source',           'value': len(runs_df),
}, {
    'metric': 'runs replaced',         'value': rt_result['runs_replaced'],
}, {
    'metric': 'runs unchanged',        'value': rt_result['runs_unchanged'],
}, {
    'metric': 'runs skipped',          'value': rt_result['runs_skipped'],
}, {
    'metric': 'output bytes',          'value': OUT_ROUND.stat().st_size,
}, {
    'metric': 'source bytes',          'value': SRC.stat().st_size,
}]))

# Vérif : re-parse et compare
rt_runs = parse_pptx(OUT_ROUND)['runs_df']
same_ids = (runs_df['span_id'].values == rt_runs['span_id'].values).all()
same_txt = (runs_df['text'].values == rt_runs['text'].values).all()
display(Markdown(f"**span_ids préservés** : `{same_ids}` &nbsp;&nbsp; **textes identiques** : `{same_txt}`"))

### 1. Round-trip identité — smoke test step 2

,metric,value
0,runs source,16
1,runs replaced,0
2,runs unchanged,16
3,runs skipped,0
4,output bytes,36416
5,source bytes,36289


**span_ids préservés** : `True` &nbsp;&nbsp; **textes identiques** : `True`

In [3]:
# 2) Démo translation FR → EN via dictionnaire (simulation, pas LLM dans la démo)
FR_TO_EN = {
    "Contrat d'assurance":                                                       "Insurance Contract",
    "Conditions G\u00e9n\u00e9rales \u2014 Pr\u00e9sentation 2026":             "General Conditions \u2014 2026 Presentation",
    "1. Garanties principales":                                                  "1. Main coverage",
    "Garantie IA (Individual Accident)":                                         "IA coverage (Individual Accident)",
    "Garantie BI (Business Interruption)":                                       "BI coverage (Business Interruption)",
    "Responsabilit\u00e9 Civile (RC)":                                           "Civil Liability (RC)",
    "Plafond global : 200 000 EUR":                                              "Overall limit: 200,000 EUR",
    "2. Franchise et d\u00e9lais":                                               "2. Deductible and deadlines",
    "La franchise applicable est de ":                                           "The applicable deductible is ",
    "300 EUR par sinistre":                                                      "300 EUR per claim",
    ". Toute d\u00e9claration doit intervenir dans un d\u00e9lai de ":           ". Any claim must be reported within ",
    "5 jours ouvr\u00e9s":                                                       "5 business days",
    ".":                                                                          ".",
    "IMPORTANT : ":                                                              "IMPORTANT: ",
    "le non-respect de ce d\u00e9lai entra\u00eene la d\u00e9ch\u00e9ance de garantie.": "failure to meet this deadline forfeits coverage.",
    "3. Tableau des garanties":                                                  "3. Coverage table",
}

# Construire le runs_df modifié (mêmes span_id, textes traduits)
translated = runs_df.copy()
translated['text'] = translated['text'].map(lambda t: FR_TO_EN.get(t, t))

n_translated = (translated['text'] != runs_df['text']).sum()
display(Markdown(f"### 2. Translation FR → EN appliquée sur {n_translated} / {len(runs_df)} runs"))

# Reconstruire le .pptx traduit
tr_result = build_pptx_document(translated, SRC, OUT_TRANS_FR)
display(pd.DataFrame([{
    'metric': 'runs replaced',  'value': tr_result['runs_replaced'],
}, {
    'metric': 'runs unchanged', 'value': tr_result['runs_unchanged'],
}, {
    'metric': 'runs skipped',   'value': tr_result['runs_skipped'],
}, {
    'metric': 'output path',    'value': str(OUT_TRANS_FR),
}, {
    'metric': 'output bytes',   'value': OUT_TRANS_FR.stat().st_size,
}]))

# Vérif : re-parse output et compare avant/après
tr_runs = parse_pptx(OUT_TRANS_FR)['runs_df']
compare_df = pd.DataFrame({
    'span_id':     runs_df['span_id'].values,
    'FR (avant)':  runs_df['text'].values,
    'EN (après)':  tr_runs['text'].values,
    'bold':        runs_df['bold'].values,
    'italic':      runs_df['italic'].values,
    'color':       runs_df['color'].values,
})
display(Markdown(f"### 3. Comparaison avant / après — tous les runs"))
display(compare_df)

# Styles préservés ?
checks = []
for col in ['font_name', 'font_size_pt', 'bold', 'italic', 'underline', 'color']:
    same = (runs_df[col].fillna('').astype(str) == tr_runs[col].fillna('').astype(str)).all()
    checks.append({'attribute': col, 'preserved': 'OK' if same else 'DIFF'})
display(Markdown('### 4. Styles préservés après traduction ?'))
display(pd.DataFrame(checks))

display(Markdown(f"\n>>> Ouvre `{OUT_TRANS_FR}` dans PowerPoint pour valider visuellement la traduction."))

### 2. Translation FR → EN appliquée sur 15 / 16 runs

,metric,value
0,runs replaced,15
1,runs unchanged,1
2,runs skipped,0
3,output path,_outputs\pptx_translation\contrat_assurance_EN_js.pptx
4,output bytes,36364


### 3. Comparaison avant / après — tous les runs

,span_id,FR (avant),EN (après),bold,italic,color
0,pp_0_0_0_0,Contrat d'assurance,Insurance Contract,False,False,NaN
1,pp_0_1_0_0,Conditions Générales — Présentation 2026,General Conditions — 2026 Presentation,False,False,NaN
2,pp_1_0_0_0,1. Garanties principales,1. Main coverage,False,False,NaN
3,pp_1_1_0_0,Garantie IA (Individual Accident),IA coverage (Individual Accident),False,False,NaN
4,pp_1_1_1_0,Garantie BI (Business Interruption),BI coverage (Business Interruption),False,False,NaN
5,pp_1_1_2_0,Responsabilité Civile (RC),Civil Liability (RC),False,False,NaN
6,pp_1_1_3_0,Plafond global : 200 000 EUR,"Overall limit: 200,000 EUR",True,False,#C0392B
7,pp_2_0_0_0,2. Franchise et délais,2. Deductible and deadlines,False,False,NaN
8,pp_2_1_0_0,La franchise applicable est de,The applicable deductible is,False,False,NaN
9,pp_2_1_0_1,300 EUR par sinistre,300 EUR per claim,True,False,NaN


### 4. Styles préservés après traduction ?

,attribute,preserved
0,font_name,OK
1,font_size_pt,OK
2,bold,OK
3,italic,OK
4,underline,OK
5,color,OK



>>> Ouvre `_outputs\pptx_translation\contrat_assurance_EN_js.pptx` dans PowerPoint pour valider visuellement la traduction.